# tokenfold quickstart (Python)The notebook version of [`quickstart.py`](quickstart.py) — same three operations, oneper cell, so you can poke at the receipts between steps.```pip install tokenfold```From a clone, build the binding instead:```maturin develop -m crates/tokenfold-py/Cargo.toml```Everything here is **lossless**: the Python binding only ever restructures data, neverdiscards it. Opt-in lossy array pruning is CLI-only today — see[`lossy_pruning.py`](lossy_pruning.py) for that.

In [ ]:
import json
from pathlib import Path

import tokenfold

# The same payload the Rust and Node examples use.
PAYLOAD = json.loads(Path("openai_payload.json").read_text())
[m["role"] for m in PAYLOAD["messages"]]

## 1. Compress a message listThe common case: you have a `messages` list bound for the Chat Completions API.`compress_messages` hands back a compressed list plus exact token accounting.

In [ ]:
result = tokenfold.compress_messages(PAYLOAD["messages"], model="gpt-4o", mode="BALANCED")

print(f"tokens:     {result.tokens_before} -> {result.tokens_after} "
      f"({result.tokens_saved} saved, {result.savings_pct:.1f}%)")
print(f"transforms: {', '.join(result.transforms_applied) or '(none applied)'}")
print(f"messages:   {len(result.messages)} message(s) ready to send")

In [ ]:
# `result.messages` is the compressed list — send it straight to your provider:
#   client.chat.completions.create(model="gpt-4o", messages=result.messages)
result.messages[-1]

## 2. Compress a raw request body`compress` is the bytes-first core API. Give it a whole request body — messages *and*the verbose tool schema — and pick the format.

In [ ]:
result = tokenfold.compress(json.dumps(PAYLOAD), format="OPENAI_JSON", mode="BALANCED")
report = result.report

# `best_effort` here means "no target_tokens to confirm against", not a failure:
# `compressed` is reserved for runs that provably met a target you asked for.
print(f"status:     {report.status}")
print(f"tokens:     {report.original_tokens} -> {report.compressed_tokens} "
      f"({report.saved_tokens} saved, {report.savings_pct:.1f}%)")
print(f"estimator:  {report.estimator.backend} (exact: {report.estimator.is_exact})")
for w in report.warnings:
    print(f"  warning: {w}")
print(f"payload:    {len(result.payload)} bytes")

## 3. Compress generic JSON dataNot every payload is a message list. `format="JSON"` compresses API responses, records,and logs: repeated keys fold into columnar form, repeated values into a dictionary. Bothare losslessly reversible — every stage is round-trip gated before it's applied.

In [ ]:
data = Path("api_response.json").read_text()
result = tokenfold.compress(data, format="JSON", mode="BALANCED")
report = result.report

applied = [t["id"] for t in report.raw["transforms"] if t["status"] == "applied"]
print(f"tokens:     {report.original_tokens} -> {report.compressed_tokens} "
      f"({report.saved_tokens} saved, {report.savings_pct:.1f}%)")
print(f"transforms: {', '.join(applied)}")
print(f"bytes:      {len(data)} -> {len(result.payload)}")

## 4. Read the whole receipttokenfold never silently drops content — you get receipts. The promoted attributes aboveare a subset; `report.raw` is the full report as a plain dict, which is where `quality`,`budget`, `cache`, `retrieval`, and the per-transform breakdown live.

In [ ]:
row = "{:<18} {:<10} {:>7} {:>7} {:>7}"
print(row.format("TRANSFORM", "STATUS", "BEFORE", "AFTER", "SAVED"))
for t in report.raw["transforms"]:
    print(row.format(t["id"], t["status"], t["tokens_before"], t["tokens_after"], t["saved_tokens"]))

In [ ]:
# Everything the transform table leaves out:
{k: v for k, v in report.raw.items() if k != "transforms"}

## Where to next- [`quickstart.py`](quickstart.py) — the same three operations as a plain script- [`quickstart.mjs`](quickstart.mjs) — the TypeScript/Node package- [`quickstart.rs`](quickstart.rs) — the embedded Rust core API- [`lossy_pruning.py`](lossy_pruning.py) — opt-in recoverable pruning, CLI-driven